# Candescence Interface Tutorial

**Purpose**: Demonstrate how to use the Candescence interface module for model inference and latent space exploration  
**Author**: Hallett Lab  
**Date**: 2026-01-29

This notebook walks through the complete workflow for using the Candescence interface:

1. Loading trained TLV models with `load_tlv_model` and `TLVModelWrapper`
2. Encoding images and exploring the latent space
3. Interactive visualization with `LatentExplorer`
4. Image display and comparison with `ImagePanel`
5. Latent space interpolation with `InterpolationTool`

## Environment

```bash
uv sync   # or, once created: source .venv/bin/activate
```

## 1. Environment Setup

In [ ]:
import sys
from pathlib import Path

# Add src to path for development (not needed if package is installed)
src_path = Path("../src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Verify imports work
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Enable inline plotting
%matplotlib inline

# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Candescence interface imports
from candescence.interface import (
    load_tlv_model,
    TLVModelWrapper,
    CandescenceModel,
)
from candescence.interface.components import (
    LatentExplorer,
    ImagePanel,
    InterpolationTool,
)

# Other candescence imports
from candescence.core.config import TLVConfig
from candescence.core.logging_config import get_logger, configure_logging
from candescence.tlv.factory import Factory
from candescence.tlv.utilities import convert_rgb_transformed_2_rgb

# Setup logging
configure_logging(level='INFO')
logger = get_logger("tutorial_interface")

print("All imports successful!")

## 2. Loading a Pre-trained Model

The interface provides `load_tlv_model()` to easily load trained TLV models. This function:
- Loads the model weights from a checkpoint
- Reads the configuration (architecture, latent dim, etc.)
- Returns a `TLVModelWrapper` that conforms to the `CandescenceModel` interface

In [ ]:
# Path to a trained model
# Using the 37-tendrils reproduction model (production quality)
model_path = Path("<refined>/0_37_tendrils_reproduction/manuscript_v1/models/model.pth")

# Alternative: use the tutorial model
# model_path = Path("<refined>/tutorial_tendril_vae/test_run_v1/models/model.pth")

if not model_path.exists():
    print(f"Model not found at {model_path}")
    print("Please train a model first using tutorial_tendril_vae.ipynb")
else:
    print(f"Found model at: {model_path}")

In [ ]:
# Load the model using the interface
model = load_tlv_model(
    model_path=model_path,
    metadata={
        'name': 'Tendril VAE - Production',
        'version': '1.0.0',
        'description': 'Hue-conditioned VAE for Candida morphology',
        'is_production': True,
    }
)

print(f"Model loaded successfully!")
print(f"\nModel info:")
for key, value in model.model_info.items():
    print(f"  {key}: {value}")

In [ ]:
# The TLVModelWrapper provides a consistent interface
print(f"\nCandescenceModel Interface:")
print(f"  Latent dimension: {model.get_latent_dim()}")
print(f"  Device: {model.device}")
print(f"  Is CandescenceModel: {isinstance(model, CandescenceModel)}")

## 3. Preparing Test Data

To demonstrate the interface, we need a dataset. We'll use the same dataset loading as in training.

In [ ]:
# Create a minimal config for data loading
config = TLVConfig(
    experiment_name="interface_tutorial",
    save_name="demo"
)

# Dataset settings (same as training)
config.strategy = 14
config.image_dimension = 128
config.image_type = 'non-wash'
config.restrict_to_day = 2
config.dataset_seed = 9954

# Use smaller subset for demo
config.train_num = 500
config.validation_num = 200
config.test_num = 200

# Conditioning (must match model)
config.conditional_variables = ['average_hue']
config.conditional_decoder_fixed_values = {'average_hue': 0.5}
config.augment_decoder_images = False
config.adjust_images = False

# System settings
config.process_priority = 19
config.num_threads = 10

In [ ]:
# Load dataset
factory = Factory(config)
factory.load_dataset()

print(f"Dataset loaded: {len(factory.dataset)} total samples")
print(f"  Training: {len(factory.dataset.train_dataset)}")
print(f"  Validation: {len(factory.dataset.validation_dataset)}")
print(f"  Test: {len(factory.dataset.test_dataset)}")

In [ ]:
# Extract the target DataFrame with metadata
target_df = factory.dataset.target_df

print(f"\nMetadata columns: {len(target_df.columns)}")
print(f"Sample columns: {list(target_df.columns)[:10]}...")

## 4. Encoding Images with the Model Interface

The `TLVModelWrapper` provides a simple `encode()` / `decode()` interface that handles:
- Device transfer (CPU/GPU)
- Conditioning variables
- Skip connection caching for U-Net architectures

In [ ]:
# Prepare a batch of images
n_samples = 100
sample_indices = np.random.choice(len(target_df), n_samples, replace=False)

# Get images and conditioning
# Note: 'transformed_image' contains the tensor, 'average_hue' is 0-255 and needs normalization
images = np.stack([target_df.iloc[i]['transformed_image'].numpy() for i in sample_indices])
hues = np.array([target_df.iloc[i]['average_hue'] / 255.0 for i in sample_indices])  # Normalize to [0, 1]

print(f"Prepared {n_samples} images")
print(f"Images shape: {images.shape}")
print(f"Hues range: [{hues.min():.3f}, {hues.max():.3f}]")

In [ ]:
# Convert to tensors
images_tensor = torch.tensor(images, dtype=torch.float32)
cond_tensor = torch.tensor(hues, dtype=torch.float32).unsqueeze(1)

print(f"Image tensor shape: {images_tensor.shape}")
print(f"Conditioning tensor shape: {cond_tensor.shape}")

In [ ]:
# Encode images using the interface
# The encode() method handles device transfer internally
latent_vectors = model.encode(images_tensor, cond_tensor)

print(f"Encoded to latent space: {latent_vectors.shape}")

In [ ]:
# Decode back to images
# Note: For Tendril VAE, decode uses cached skip connections from encode
decoder_cond = torch.full((n_samples, 1), 0.5)  # Fixed hue for decoder
reconstructed = model.decode(latent_vectors, decoder_cond)

print(f"Reconstructed images shape: {reconstructed.shape}")

In [ ]:
# Convert latent vectors to numpy for further analysis
embeddings = latent_vectors.cpu().numpy()
print(f"Embeddings array shape: {embeddings.shape}")

## 5. LatentExplorer: Interactive Visualization

The `LatentExplorer` component provides interactive 2D visualization of the latent space using dimensionality reduction (UMAP, t-SNE, or PCA).

In [ ]:
# Prepare metadata DataFrame for the explorer
# Note: We normalize HSV values to [0, 1] for consistency
explorer_metadata = target_df.iloc[sample_indices][['id', 'colony_size']].copy().reset_index(drop=True)
explorer_metadata['average_hue'] = hues  # Already normalized
explorer_metadata['average_saturation'] = np.array([
    target_df.iloc[i]['average_saturation'] / 255.0 for i in sample_indices
])
explorer_metadata['average_value'] = np.array([
    target_df.iloc[i]['average_value'] / 255.0 for i in sample_indices
])

print(f"Metadata for explorer:")
display(explorer_metadata.head())

In [ ]:
# Create the LatentExplorer
explorer = LatentExplorer(
    embeddings=embeddings,
    metadata_df=explorer_metadata,
    model=model,
    reduction_method='pca',  # Start with PCA (fastest)
    random_state=42
)

print(f"LatentExplorer initialized:")
print(f"  Samples: {explorer.n_samples}")
print(f"  Latent dim: {explorer.latent_dim}")

In [ ]:
# Compute the 2D projection
coords_2d = explorer.compute_projection()

print(f"2D projection shape: {coords_2d.shape}")
print(f"X range: [{coords_2d[:, 0].min():.2f}, {coords_2d[:, 0].max():.2f}]")
print(f"Y range: [{coords_2d[:, 1].min():.2f}, {coords_2d[:, 1].max():.2f}]")

In [ ]:
# Create interactive Plotly figure (requires plotly: pip install plotly)
try:
    fig = explorer.create_figure(
        color_by='average_hue',
        title='Latent Space Colored by Hue (PCA)',
        point_size=10,
        opacity=0.8
    )
    fig.show()
except ImportError:
    print("Plotly not installed. Install with: pip install plotly")
    print("Using matplotlib fallback in next cell.")

In [ ]:
# Matplotlib version for static plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Color by different features
for ax, feature in zip(axes, ['average_hue', 'average_saturation', 'colony_size']):
    scatter = ax.scatter(
        coords_2d[:, 0], 
        coords_2d[:, 1],
        c=explorer_metadata[feature],
        cmap='viridis',
        s=30,
        alpha=0.7
    )
    plt.colorbar(scatter, ax=ax, label=feature)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title(f'Colored by {feature}')

plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Find nearest neighbors to a specific point
test_x, test_y = 0.0, 0.0
nearest_indices = explorer.get_nearest_index(test_x, test_y, k=5)

print(f"5 nearest neighbors to ({test_x}, {test_y}):")
for idx in nearest_indices:
    meta = explorer.get_metadata_at_index(idx)
    x, y = explorer.get_2d_coords_at_index(idx)
    print(f"  Index {idx}: ({x:.2f}, {y:.2f}), hue={meta['average_hue']:.3f}")

## 6. ImagePanel: Display and Comparison

The `ImagePanel` component handles image display, format conversion, and comparisons.

In [ ]:
# Create ImagePanel
panel = ImagePanel(default_size=(256, 256))

# Get a sample image and reconstruction
sample_idx = 0
original_img = images[sample_idx]
reconstructed_img = reconstructed[sample_idx].cpu().numpy()

print(f"Original shape: {original_img.shape}")
print(f"Reconstructed shape: {reconstructed_img.shape}")

In [ ]:
# Prepare images for display (handles format conversion)
orig_display = panel.prepare_image(original_img)
recon_display = panel.prepare_image(reconstructed_img)

print(f"Display format - Original: {orig_display.shape}, dtype={orig_display.dtype}")
print(f"Display format - Reconstructed: {recon_display.shape}, dtype={recon_display.dtype}")

In [ ]:
# Display side-by-side comparison
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i in range(5):
    # Original
    orig = panel.prepare_image(images[i])
    axes[0, i].imshow(orig)
    axes[0, i].set_title(f"Original {i}")
    axes[0, i].axis('off')
    
    # Reconstructed
    recon = panel.prepare_image(reconstructed[i].cpu().numpy())
    axes[1, i].imshow(recon)
    axes[1, i].set_title(f"Reconstructed {i}")
    axes[1, i].axis('off')

plt.suptitle("Original vs Reconstructed Images", fontsize=14)
plt.tight_layout()
display(fig)
plt.close(fig)

## 7. InterpolationTool: Latent Space Interpolation

The `InterpolationTool` generates smooth transitions between points in latent space, useful for understanding how the model represents different morphologies.

In [ ]:
# Prepare conditioning for interpolation
# Shape: (n_samples, cond_dim)
conditioning = hues.reshape(-1, 1)

# Create InterpolationTool
# Pass images so it can re-encode to set up skip connections correctly
interp_tool = InterpolationTool(
    model=model,
    embeddings=embeddings,
    conditioning=conditioning,
    images=images  # Required for U-Net architectures with skip connections
)

print(f"InterpolationTool ready with {len(embeddings)} embeddings")

## 7a. Interactive Latent Space Explorer

Three simple functions to explore the latent space:

1. **`show_latent_scatter(coords_2d, metadata_df)`** - Shows the scatter plot with point indices labeled
2. **`show_image_at_index(idx, ...)`** - Shows the real image at index `idx`
3. **`show_interpolation(idx_a, idx_b, ...)`** - Shows interpolation between two points

**How to use:**
- Run the cells below in order
- Look at the scatter plot to find interesting indices
- Change the numbers in the function calls and re-run to explore different points

In [ ]:
# Simple functions for interactive exploration
from IPython.display import display

def show_image_at_index(idx, images, panel, explorer):
    """Display the real image at a given index."""
    print(f"Showing image at index {idx}...")
    if idx < 0 or idx >= len(images):
        print(f"Index {idx} out of range (0-{len(images)-1})")
        return
    
    meta = explorer.get_metadata_at_index(idx)
    hue = meta.get('average_hue', 'N/A')
    if isinstance(hue, (int, float)):
        print(f"Hue = {hue:.3f}")
    
    fig, ax = plt.subplots(figsize=(5, 5))
    img = panel.prepare_image(images[idx])
    ax.imshow(img)
    ax.set_title(f"Image at index {idx}")
    ax.axis('off')
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    print("Done!")


def show_interpolation(idx_a, idx_b, images, conditioning, explorer, panel, model, n_steps=7):
    """Show interpolation between two points."""
    print(f"Interpolating from index {idx_a} to {idx_b}...")
    if idx_a < 0 or idx_a >= len(images) or idx_b < 0 or idx_b >= len(images):
        print(f"Indices must be in range 0-{len(images)-1}")
        return
    
    z_a = explorer.embeddings[idx_a]
    z_b = explorer.embeddings[idx_b]
    cond_a = conditioning[idx_a]
    cond_b = conditioning[idx_b]
    
    # Set up skip connections
    ref_img = torch.tensor(images[idx_a:idx_a+1], dtype=torch.float32)
    ref_cond = torch.tensor(conditioning[idx_a:idx_a+1], dtype=torch.float32)
    _ = model.encode(ref_img, ref_cond)
    
    # Generate interpolation
    alphas = np.linspace(0, 1, n_steps)
    interp_images = []
    
    print(f"Generating {n_steps} interpolation steps...")
    for alpha in alphas:
        z_interp = (1 - alpha) * z_a + alpha * z_b
        cond_interp = (1 - alpha) * cond_a + alpha * cond_b
        z_tensor = torch.tensor(z_interp, dtype=torch.float32).unsqueeze(0)
        cond_tensor = torch.tensor(cond_interp, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            decoded = model.decode(z_tensor, cond_tensor)
        interp_images.append(decoded.cpu().numpy().squeeze())
    
    # Plot 1: Real endpoints
    print("Showing endpoint images...")
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(panel.prepare_image(images[idx_a]))
    axes[0].set_title(f"Point A: Real Image (index {idx_a})")
    axes[0].axis('off')
    axes[1].imshow(panel.prepare_image(images[idx_b]))
    axes[1].set_title(f"Point B: Real Image (index {idx_b})")
    axes[1].axis('off')
    plt.suptitle("Endpoint Real Images", fontsize=14)
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    
    # Plot 2: Interpolation filmstrip
    print("Showing interpolation filmstrip...")
    fig, axes = plt.subplots(1, n_steps, figsize=(2.5 * n_steps, 3))
    for i, (ax, img) in enumerate(zip(axes, interp_images)):
        ax.imshow(panel.prepare_image(img))
        ax.set_title(f"α={alphas[i]:.2f}")
        ax.axis('off')
    plt.suptitle("Reconstructed Interpolation", fontsize=14)
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    
    # Plot 3: Path on latent space
    print("Showing path in latent space...")
    coords_2d = explorer.compute_projection()
    color_vals = explorer.metadata_df.get('average_hue', np.zeros(len(coords_2d)))
    
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(coords_2d[:, 0], coords_2d[:, 1], c=color_vals, cmap='viridis', s=20, alpha=0.3)
    ax.scatter(coords_2d[idx_a, 0], coords_2d[idx_a, 1], 
              c='red', s=300, marker='*', edgecolors='black', linewidths=2, label=f'A (idx {idx_a})')
    ax.scatter(coords_2d[idx_b, 0], coords_2d[idx_b, 1], 
              c='blue', s=300, marker='*', edgecolors='black', linewidths=2, label=f'B (idx {idx_b})')
    ax.plot([coords_2d[idx_a, 0], coords_2d[idx_b, 0]],
           [coords_2d[idx_a, 1], coords_2d[idx_b, 1]], 'k--', linewidth=2, alpha=0.7)
    ax.legend()
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    ax.set_title('Interpolation Path in Latent Space')
    plt.tight_layout()
    display(fig)
    plt.close(fig)
    print("Done!")

print("Functions defined: show_image_at_index(), show_interpolation()")

In [ ]:
# Step 1: View the latent space scatter plot (indices are labeled)
print("Computing projection...")
coords_2d = explorer.compute_projection()
print(f"Projection computed: {coords_2d.shape}")

print("Creating scatter plot...")
color_vals = explorer.metadata_df.get('average_hue', np.zeros(len(coords_2d)))
print(f"Color values: {len(color_vals)} points, range [{color_vals.min():.2f}, {color_vals.max():.2f}]")

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(coords_2d[:, 0], coords_2d[:, 1], 
                    c=color_vals, cmap='viridis', s=40, alpha=0.7)

# Label every 5th point
for i in range(0, len(coords_2d), 5):
    ax.annotate(str(i), (coords_2d[i, 0], coords_2d[i, 1]), fontsize=6, alpha=0.5)

plt.colorbar(scatter, ax=ax, label='Hue')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title('Latent Space - use indices to select points')
plt.tight_layout()
display(fig)
plt.close(fig)
print("Done!")

In [ ]:
# Step 2: View an image at a specific index (change the number to explore)
# First re-run the function definitions cell above if you haven't!
show_image_at_index(15, images, panel, explorer)

In [ ]:
# Step 3: Interpolate between two points (change idx_a and idx_b to explore)
show_interpolation(idx_a=5, idx_b=40, images=images, conditioning=conditioning, 
                   explorer=explorer, panel=panel, model=model)

In [ ]:
# Choose two points to interpolate between
# Find points with different hue values for interesting interpolation
hue_sorted_indices = np.argsort(hues)
point_a = hue_sorted_indices[0]   # Lowest hue
point_b = hue_sorted_indices[-1]  # Highest hue

print(f"Point A: index {point_a}, hue={hues[point_a]:.3f}")
print(f"Point B: index {point_b}, hue={hues[point_b]:.3f}")

In [ ]:
# Generate interpolation with conditioning interpolated as well
interp_images = interp_tool.interpolate(
    index_a=point_a,
    index_b=point_b,
    n_steps=7,
    conditioning_strategy='interpolate'
)

print(f"Generated {len(interp_images)} interpolation steps")
print(f"Each image shape: {interp_images[0].shape}")

In [ ]:
# Display interpolation as filmstrip
n_steps = len(interp_images)
fig, axes = plt.subplots(1, n_steps, figsize=(2.5 * n_steps, 3))

for i, (ax, img) in enumerate(zip(axes, interp_images)):
    display_img = panel.prepare_image(img)
    ax.imshow(display_img)
    alpha = i / (n_steps - 1)
    ax.set_title(f"α={alpha:.2f}", fontsize=10)
    ax.axis('off')

plt.suptitle(f"Interpolation: Hue {hues[point_a]:.2f} → {hues[point_b]:.2f}", fontsize=12)
plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Compare different conditioning strategies
strategies = ['interpolate', 'first', 'second']

fig, axes = plt.subplots(len(strategies), 7, figsize=(17, 7))

for row, strategy in enumerate(strategies):
    images_strat = interp_tool.interpolate(
        index_a=point_a,
        index_b=point_b,
        n_steps=7,
        conditioning_strategy=strategy
    )
    
    for col, img in enumerate(images_strat):
        display_img = panel.prepare_image(img)
        axes[row, col].imshow(display_img)
        axes[row, col].axis('off')
        
        if col == 0:
            axes[row, col].set_ylabel(strategy, fontsize=11)
        if row == 0:
            alpha = col / 6
            axes[row, col].set_title(f"α={alpha:.2f}", fontsize=10)

plt.suptitle("Interpolation with Different Conditioning Strategies", fontsize=14)
plt.tight_layout()
display(fig)
plt.close(fig)

## 8. Latent Space Walk

Exploring individual latent dimensions to understand what the model has learned.

In [ ]:
# Analyze correlation of latent dimensions with hue
correlations = []
for dim in range(embeddings.shape[1]):
    corr = np.corrcoef(embeddings[:, dim], hues)[0, 1]
    correlations.append(corr)

correlations = np.array(correlations)

# Find most correlated dimensions
top_positive = np.argsort(correlations)[-3:][::-1]
top_negative = np.argsort(correlations)[:3]

print("Most positively correlated with hue:")
for idx in top_positive:
    print(f"  Dim {idx}: r = {correlations[idx]:.3f}")

print("\nMost negatively correlated with hue:")
for idx in top_negative:
    print(f"  Dim {idx}: r = {correlations[idx]:.3f}")

In [ ]:
# Plot correlation distribution
fig, ax = plt.subplots(figsize=(12, 4))

colors = ['red' if c < -0.3 else 'blue' if c > 0.3 else 'gray' for c in correlations]
ax.bar(range(len(correlations)), correlations, color=colors, alpha=0.7)
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=0.3, color='blue', linestyle='--', alpha=0.5)
ax.axhline(y=-0.3, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Latent Dimension')
ax.set_ylabel('Correlation with Hue')
ax.set_title('Latent Dimension Correlations with Average Hue')

plt.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Latent walk: vary specific dimensions
# Use a reference sample
ref_idx = 50
ref_embedding = embeddings[ref_idx]
ref_cond = conditioning[ref_idx]

# First encode the reference image to set skip connections
ref_img_tensor = torch.tensor(images[ref_idx:ref_idx+1], dtype=torch.float32)
ref_cond_tensor = torch.tensor(ref_cond, dtype=torch.float32).unsqueeze(0)
_ = model.encode(ref_img_tensor, ref_cond_tensor)

# Dimensions to walk
dims_to_walk = [0, top_positive[0], top_negative[0], 64]
walk_range = np.linspace(-3, 3, 7)

decoder_cond = torch.tensor([[0.5]], dtype=torch.float32).to(model.device)

fig, axes = plt.subplots(len(dims_to_walk), len(walk_range), figsize=(2.5 * len(walk_range), 2.5 * len(dims_to_walk)))

for row, dim in enumerate(dims_to_walk):
    for col, delta in enumerate(walk_range):
        z_mod = ref_embedding.copy()
        z_mod[dim] = z_mod[dim] + delta
        
        z_tensor = torch.tensor(z_mod, dtype=torch.float32).unsqueeze(0).to(model.device)
        
        with torch.no_grad():
            decoded = model.decode(z_tensor, decoder_cond)
        
        display_img = panel.prepare_image(decoded.cpu().numpy().squeeze())
        axes[row, col].imshow(display_img)
        axes[row, col].axis('off')
        
        if col == 0:
            corr_label = f"(r={correlations[dim]:.2f})"
            axes[row, col].set_ylabel(f"Dim {dim}\n{corr_label}", fontsize=10)
        if row == 0:
            axes[row, col].set_title(f"Δ={delta:.1f}", fontsize=10)

plt.suptitle("Latent Walk: Effect of Individual Dimensions", fontsize=14)
plt.tight_layout()
display(fig)
plt.close(fig)

## 9. Summary

This tutorial demonstrated the key components of the Candescence interface:

### Model Loading
- `load_tlv_model()`: Easy loading of trained models from checkpoints
- `TLVModelWrapper`: Unified interface with `encode()` / `decode()` methods
- `CandescenceModel`: Abstract interface ensuring consistent API across model types

### Visualization Components
- `LatentExplorer`: 2D latent space visualization with PCA/UMAP/t-SNE
- `ImagePanel`: Image format conversion and display utilities
- `InterpolationTool`: Smooth interpolation between latent points

### Analysis
- Latent dimension correlations with conditioning variables
- Latent walks to understand learned representations
- Conditioning strategy comparisons

### Note on Data Utilities
The interface also provides data cleaning utilities (`standardize_missing_values`, `clean_non_ascii`, `DataQualityReport`) which are useful when preprocessing external metadata files (e.g., Calb_Master.tsv), but are not directly used in the TLV inference workflow shown here.

In [ ]:
print("\n" + "="*60)
print("INTERFACE TUTORIAL COMPLETE")
print("="*60)
print(f"\nModel: {model.model_info['name']}")
print(f"Samples processed: {n_samples}")
print(f"Latent dimension: {model.get_latent_dim()}")
print(f"Device: {model.device}")